# 208 — GraphAgent
Specialist agent for graph exploration and entity context retrieval.
Runs each supported task, logs events to Neo4j, and optionally generates
an AI summary.

Supported tasks: `entity_lookup`, `company_profile`, `expand_ownership`,
`shared_address_check`, `sic_context`.

In [1]:
import sys
sys.path.insert(0, "..")

In [2]:
import uuid
import logging
import pandas as pd

from src.config import get_neo4j_settings
from src.storage.neo4j_repository import Neo4jRepository
from src.storage.trace_repository import TraceRepository
from src.tracing.trace_service import TraceService
from src.clients.mcp_tool_client import MCPToolClient
from src.tools.trace_tools import TraceTools
from src.agents.graph_agent import GraphAgent
from src.domain.models import InvestigationRequest, UserContext

logging.basicConfig(
    level=logging.INFO,
    format="%(name)-32s %(levelname)-8s %(message)s",
)

settings = get_neo4j_settings()
neo4j    = Neo4jRepository(**vars(settings))
neo4j.test_connection()

repo          = TraceRepository(neo4j)
trace_service = TraceService(repo)
mcp_client    = MCPToolClient()
trace_tools   = TraceTools(repo)
print("Connected")

Connected


## Run without AI client
All five tasks against a single trace.  
Summaries are the deterministic tool summaries — no LLM involved.

In [3]:
COMPANY = "TELEFONICA O2 HOLDINGS LIMITED"

agent_no_ai = GraphAgent(mcp_client, trace_service)
print(f"agent : {agent_no_ai.name}")
print(f"ai    : {agent_no_ai._ai_client}")

agent : graph-agent
ai    : None


In [4]:
request = InvestigationRequest(
    entity_name=COMPANY,
    context=UserContext(
        user_id="analyst-001",
        session_id=str(uuid.uuid4()),
        metadata={"mode": "interactive"},
    ),
    request_id=str(uuid.uuid4()),
)
trace_no_ai = trace_service.start_trace(request, request.context)
print(f"trace_id: {trace_no_ai.request_id}")

trace_id: 88743d25-a42b-40ac-b270-3b68eeb8829f


In [5]:
tasks = [
    ("entity_lookup",       {"company_name": COMPANY}),
    ("company_profile",     {"company_name": COMPANY}),
    ("expand_ownership",    {"company_name": COMPANY, "max_depth": 3}),
    ("shared_address_check",{"company_name": COMPANY}),
    ("sic_context",         {"company_name": COMPANY}),
]

results_no_ai = {}
for task, ctx in tasks:
    r = agent_no_ai.run(task, ctx, trace_no_ai)
    results_no_ai[task] = r
    status = "✓" if r.success else "✗"
    print(f"[{status}] {task:<25}  {r.summary[:90]}")
    print(f"         tools: {r.tools_used}")

[✓] entity_lookup              Found 10 company match(es) for 'TELEFONICA O2 HOLDINGS LIMITED'. Top match: 'TELEFONICA O2
         tools: ['entity_lookup']
[✓] company_profile            TELEFONICA O2 HOLDINGS LIMITED (#05310128, Active). SIC codes: 61900 (Other telecommunicat
         tools: ['company_profile']
[✓] expand_ownership           Found 1 ownership hop(s) across depths [1] for 'TELEFONICA O2 HOLDINGS LIMITED'. All chain
         tools: ['expand_ownership']
[✓] shared_address_check       'TELEFONICA O2 HOLDINGS LIMITED' shares its address (BN99 3HH) with 131 other companies (1
         tools: ['shared_address_check']
[✓] sic_context                'TELEFONICA O2 HOLDINGS LIMITED' operates under SIC code(s): 61900 (Other telecommunicatio
         tools: ['sic_context']


### Inspect individual results

In [6]:
r = results_no_ai["company_profile"]
print(f"success  : {r.success}")
print(f"summary  : {r.summary}")
if r.findings.get("company_profile"):
    profile = r.findings["company_profile"]
    print(f"\ncompany  : {profile['company']}")
    print(f"address  : {profile['address']}")
    print(f"sic_codes: {profile['sic_codes']}")
    print(f"owners   : {profile['direct_owners']}")

success  : True
summary  : TELEFONICA O2 HOLDINGS LIMITED (#05310128, Active). SIC codes: 61900 (Other telecommunications activities), 74990 (Non-trading company). Direct owners: Telefonica S A.

company  : {'name': 'TELEFONICA O2 HOLDINGS LIMITED', 'company_number': '05310128', 'status': 'Active', 'country_of_origin': 'United Kingdom'}
address  : {'address_line_1': 'HIGHDOWN HOUSE', 'address_line_2': 'YEOMAN WAY', 'post_town': 'WORTHING', 'county': 'WEST SUSSEX', 'post_code': 'BN99 3HH', 'country': 'UNITED KINGDOM'}
sic_codes: [{'sic_code': '61900', 'sic_description': 'Other telecommunications activities'}, {'sic_code': '74990', 'sic_description': 'Non-trading company'}]
owners   : [{'owner_name': 'Telefonica S A', 'owner_labels': ['Company'], 'ownership_pct_min': 75, 'ownership_pct_max': 100, 'ownership_controls': ['ownership-of-shares-75-to-100-percent']}]


In [7]:
r = results_no_ai["expand_ownership"]
print(f"success  : {r.success}")
print(f"summary  : {r.summary}")
if r.findings.get("expand_ownership"):
    ow = r.findings["expand_ownership"]
    print(f"\npaths ({len(ow['paths'])}) — first 3:")
    for p in ow["paths"][:3]:
        print(f"  {p}")
    print(f"\nUBOs: {ow['ultimate_owners']}")

success  : True
summary  : Found 1 ownership hop(s) across depths [1] for 'TELEFONICA O2 HOLDINGS LIMITED'. All chains terminate at corporate entities — no individual UBOs found.

paths (1) — first 3:
  {'path_depth': 1, 'hop': 1, 'from_name': 'Telefonica S A', 'from_labels': ['Company'], 'ownership_pct_min': 75, 'ownership_pct_max': 100, 'ownership_controls': ['ownership-of-shares-75-to-100-percent'], 'to_name': 'TELEFONICA O2 HOLDINGS LIMITED', 'to_labels': ['Company']}

UBOs: []


### Trace events written so far

In [8]:
trace_service.finalize_trace(trace_no_ai, final_summary=results_no_ai["company_profile"].summary)

r = trace_tools.retrieve_trace(trace_no_ai.request_id)
print(f"[{'✓' if r.success else '✗'}] {r.summary}")
if r.data:
    df = pd.DataFrame(r.data["events"])
    display(df[["event_number", "agent_name", "tool_name", "input_summary", "output_summary", "decision"]])

[✓] Trace '88743d25-a42b-40ac-b270-3b68eeb8829f' for 'TELEFONICA O2 HOLDINGS LIMITED': 5 event(s).


,event_number,agent_name,tool_name,input_summary,output_summary,decision
0,1,graph-agent,entity_lookup,name=TELEFONICA O2 HOLDINGS LIMITED,Found 10 company match(es) for 'TELEFONICA O2 ...,result available for downstream agents
1,2,graph-agent,company_profile,company_name=TELEFONICA O2 HOLDINGS LIMITED,"TELEFONICA O2 HOLDINGS LIMITED (#05310128, Act...",result available for downstream agents
2,3,graph-agent,expand_ownership,"company_name=TELEFONICA O2 HOLDINGS LIMITED, m...",Found 1 ownership hop(s) across depths [1] for...,result available for downstream agents
3,4,graph-agent,shared_address_check,company_name=TELEFONICA O2 HOLDINGS LIMITED,'TELEFONICA O2 HOLDINGS LIMITED' shares its ad...,result available for downstream agents
4,5,graph-agent,sic_context,company_name=TELEFONICA O2 HOLDINGS LIMITED,'TELEFONICA O2 HOLDINGS LIMITED' operates unde...,result available for downstream agents


## Run with AI client
Repeats `company_profile` on a **separate trace** with an `AnthropicClient` so we can
compare the deterministic summary vs the AI-enriched one, and inspect both traces
independently.

Skipped automatically if `ANTHROPIC_API_KEY` is not set.

In [9]:
import os

ai_client = None
if os.getenv("ANTHROPIC_API_KEY"):
    from src.config import get_anthropic_settings
    from src.clients.anthropic_client import AnthropicClient
    ai_settings = get_anthropic_settings()
    ai_client   = AnthropicClient(settings=ai_settings)
    print(f"AI client ready — model: {ai_settings.model_haiku}")
else:
    print("ANTHROPIC_API_KEY not set — AI enrichment cell will be skipped")

AI client ready — model: claude-haiku-4-5-20251001


In [10]:
if ai_client is None:
    trace_ai = None
    print("Skipped — no AI client")
else:
    agent_ai = GraphAgent(mcp_client, trace_service, ai_client)

    request_ai = InvestigationRequest(
        entity_name=COMPANY,
        context=UserContext(
            user_id="analyst-001",
            session_id=str(uuid.uuid4()),
            metadata={"mode": "interactive"},
        ),
        request_id=str(uuid.uuid4()),
    )
    trace_ai = trace_service.start_trace(request_ai, request_ai.context)
    print(f"trace_id (no-AI): {trace_no_ai.request_id}")
    print(f"trace_id (AI)   : {trace_ai.request_id}")
    print()

    results_ai = {}
    for task, ctx in tasks:
        r = agent_ai.run(task, ctx, trace_ai)
        results_ai[task] = r
        status = "✓" if r.success else "✗"
        print(f"[{status}] {task:<25}  {r.summary[:90]}")
        print(f"         tools: {r.tools_used}")

trace_id (no-AI): 88743d25-a42b-40ac-b270-3b68eeb8829f
trace_id (AI)   : 4c6e75f7-f619-4154-accc-e518f99a466c

[✓] entity_lookup              Found 10 company match(es) for 'TELEFONICA O2 HOLDINGS LIMITED'. Top match: 'TELEFONICA O2
         tools: ['entity_lookup']


[03/23/26 19:24:11] INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 ]8;id=762702;file:///Users/emilpastor/opt/miniconda3/envs/ai-governance/lib/python3.11/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=912147;file:///Users/emilpastor/opt/miniconda3/envs/ai-governance/lib/python3.11/site-packages/httpx/_client.py#1025\1025]8;;\
                             OK"                                                                                   

[✓] company_profile            Telefonica O2 Holdings Limited is an active UK telecommunications company ultimately owned
         tools: ['company_profile']


[03/23/26 19:24:13] INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 ]8;id=890466;file:///Users/emilpastor/opt/miniconda3/envs/ai-governance/lib/python3.11/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=438643;file:///Users/emilpastor/opt/miniconda3/envs/ai-governance/lib/python3.11/site-packages/httpx/_client.py#1025\1025]8;;\
                             OK"                                                                                   

[✓] expand_ownership           TELEFONICA O2 HOLDINGS LIMITED's ownership structure terminates at corporate entities with
         tools: ['expand_ownership']


[03/23/26 19:24:14] INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 ]8;id=183729;file:///Users/emilpastor/opt/miniconda3/envs/ai-governance/lib/python3.11/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=691096;file:///Users/emilpastor/opt/miniconda3/envs/ai-governance/lib/python3.11/site-packages/httpx/_client.py#1025\1025]8;;\
                             OK"                                                                                   

[✓] shared_address_check       Telefonica O2 Holdings Limited operates from a shared address with 126 other active compan
         tools: ['shared_address_check']
[✓] sic_context                'TELEFONICA O2 HOLDINGS LIMITED' operates under SIC code(s): 61900 (Other telecommunicatio
         tools: ['sic_context']


In [11]:
if trace_ai is not None:
    print("--- company_profile comparison ---")
    print(f"No AI : {results_no_ai['company_profile'].summary}")
    print(f"AI    : {results_ai['company_profile'].summary}")

--- company_profile comparison ---
No AI : TELEFONICA O2 HOLDINGS LIMITED (#05310128, Active). SIC codes: 61900 (Other telecommunications activities), 74990 (Non-trading company). Direct owners: Telefonica S A.
AI    : Telefonica O2 Holdings Limited is an active UK telecommunications company ultimately owned by Telefonica S.A. that operates non-trading activities alongside other telecommunications services.


In [12]:
if trace_ai is not None:
    trace_service.finalize_trace(trace_ai, final_summary=results_ai["company_profile"].summary)

    r = trace_tools.retrieve_trace(trace_ai.request_id)
    print(f"[{'✓' if r.success else '✗'}] {r.summary}")
    if r.data:
        df = pd.DataFrame(r.data["events"])
        display(df[["event_number", "agent_name", "event_type", "tool_name", "decision", "why"]])

[✓] Trace '4c6e75f7-f619-4154-accc-e518f99a466c' for 'TELEFONICA O2 HOLDINGS LIMITED': 8 event(s).


,event_number,agent_name,event_type,tool_name,decision,why
0,1,graph-agent,tool_returned,entity_lookup,result available for downstream agents,
1,2,graph-agent,tool_returned,company_profile,result available for downstream agents,
2,3,graph-agent,agent_reasoning,,AI summary generated for task 'company_profile...,Telefonica O2 Holdings Limited is an active UK...
3,4,graph-agent,tool_returned,expand_ownership,result available for downstream agents,
4,5,graph-agent,agent_reasoning,,AI summary generated for task 'expand_ownershi...,TELEFONICA O2 HOLDINGS LIMITED's ownership str...
5,6,graph-agent,tool_returned,shared_address_check,result available for downstream agents,
6,7,graph-agent,agent_reasoning,,AI summary generated for task 'shared_address_...,Telefonica O2 Holdings Limited operates from a...
7,8,graph-agent,tool_returned,sic_context,result available for downstream agents,


## Error cases

In [13]:
# Unknown task
r_bad = agent_no_ai.run("not_a_task", {"company_name": COMPANY}, trace_no_ai)
print(f"unknown task  → success={r_bad.success}  error={r_bad.error}")

# Missing company_name
r_empty = agent_no_ai.run("company_profile", {}, trace_no_ai)
print(f"missing name  → success={r_empty.success}  error={r_empty.error}")

unknown task  → success=False  error=Unknown task 'not_a_task'. Supported tasks: company_profile, entity_lookup, expand_ownership, shared_address_check, sic_context.
missing name  → success=False  error=context must include a non-empty 'company_name'.


## Cleanup — delete all traces created in this notebook

In [14]:
for trace_id in [trace_no_ai.request_id] + ([trace_ai.request_id] if trace_ai else []):
    neo4j.run_query(
        """
        MATCH (t:InvestigationTrace {trace_id: $trace_id})
        OPTIONAL MATCH (t)-[:HAS_EVENT]->(e:TraceEvent)
        DETACH DELETE t, e
        """,
        {"trace_id": trace_id},
    )
    print(f"Deleted {trace_id}")

Deleted 88743d25-a42b-40ac-b270-3b68eeb8829f
Deleted 4c6e75f7-f619-4154-accc-e518f99a466c


In [15]:
neo4j.close()
print("Driver closed")

Driver closed
